In [ ]:
# Database Connection & Data Verification

## Purpose
This notebook establishes a connection to the PostgreSQL database containing HDX (Humanitarian Data Exchange) datasets for South Sudan and verifies that all raw data has been loaded correctly.

## What This Notebook Does
1. Connects to PostgreSQL using environment variables
2. Tests the database connection
3. Lists all schemas, tables, and row counts
4. Previews the first 5 rows of each table

## Data Sources Verified
- Population estimates (2024)
- Health facilities
- Health facility types (reference/lookup table)
- Education facilities
- JIAF (Joint Inter-Agency Analysis Framework) data for South Sudan 2026

## Output
A verified database connection with confirmed data loads, ready for data profiling in Notebook 2.

In [2]:
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load environment variables from .env file
load_dotenv()

# Get database credentials from environment variables
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Create database connection string
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create database engine
engine = create_engine(DATABASE_URL)

print("Environment variables loaded successfully")
print(f"Connected to: {DB_NAME} on {DB_HOST}:{DB_PORT}")

Environment variables loaded successfully
Connected to: humanitarian_db on localhost:5432


In [3]:
# Test PostgreSQL version
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    version = result.fetchone()[0]
    print(f"PostgreSQL version: {version}")

# Test current database
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    db_name = result.fetchone()[0]
    print(f"Connected to database: {db_name}")

print("\nDatabase connection successful!")

PostgreSQL version: PostgreSQL 17.10 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit
Connected to database: humanitarian_db

Database connection successful!


In [4]:
# Query to list all schemas, tables, and row counts
query = """
SELECT 
    table_schema,
    table_name,
    pg_stat_user_tables.n_live_tup AS row_count
FROM 
    information_schema.tables
LEFT JOIN 
    pg_stat_user_tables 
    ON information_schema.tables.table_name = pg_stat_user_tables.relname
WHERE 
    table_schema NOT IN ('information_schema', 'pg_catalog')
    AND table_type = 'BASE TABLE'
ORDER BY 
    table_schema, table_name;
"""

# Execute query and display results as DataFrame
with engine.connect() as conn:
    df_tables = pd.read_sql(query, conn)

# Display the results
print(" Database Tables Overview")
print("-" * 50)
df_tables

 Database Tables Overview
--------------------------------------------------


,table_schema,table_name,row_count
0,gis,country_boundary,1
1,gis,county_boundary,79
2,gis,payam_boundary,512
3,gis,state_boundary,10
4,public,spatial_ref_sys,8500
5,raw_data,education_facilities,573
6,raw_data,health_facilities,1988
7,raw_data,health_facility_type,1513
8,raw_data,jiaf_south_sudan_2026,239
9,raw_data,population_estimates_2024,80


In [5]:
# List of tables to preview
tables = [
    'raw_data.education_facilities',
    'raw_data.health_facilities',
    'raw_data.health_facility_type',
    'raw_data.jiaf_south_sudan_2026',
    'raw_data.population_estimates_2024'
]

# Preview each table
for table in tables:
    print(f"\n{'='*60}")
    print(f"Preview: {table}")
    print('='*60)
    
    query = f"SELECT * FROM {table} LIMIT 5;"
    
    with engine.connect() as conn:
        df_preview = pd.read_sql(query, conn)
    
    print(f"Columns: {', '.join(df_preview.columns)}")
    print(f"Shape: {df_preview.shape[0]} rows x {df_preview.shape[1]} columns")
    print("\nPreview:")
    display(df_preview)


Preview: raw_data.education_facilities
Columns: id, name, name_en, amenity, building, operator_type, capacity_persons, addr_full, addr_city, source, adm0_pcode, adm0_name, adm1_pcode, adm1_name, adm2_pcode, adm2_name, adm3_pcode, adm3_name, adm4_pcode, adm4_name, name_latin, geometry
Shape: 5 rows x 22 columns

Preview:


,id,name,name_en,amenity,building,operator_type,capacity_persons,addr_full,addr_city,source,...,adm1_pcode,adm1_name,adm2_pcode,adm2_name,adm3_pcode,adm3_name,adm4_pcode,adm4_name,name_latin,geometry
0,way/974164633,Africano Mande Academy,None,school,None,None,None,None,Maridi,None,...,SS10,Western Equatoria,SS1003,Maridi,SS100304,Maridi,None,None,Africano Mande Academy,0103000020E61000000100000011000000A62089A8D373...
1,way/974164631,Maridi Teacher Training Institute,None,school,None,None,None,None,Maridi,None,...,SS10,Western Equatoria,SS1003,Maridi,SS100304,Maridi,None,None,Maridi Teacher Training Institute,0103000020E610000001000000070000000A7EC0A84973...
2,way/974164630,AMREF Nurse & Midwifery Training,None,school,None,None,None,None,Maridi,None,...,SS10,Western Equatoria,SS1003,Maridi,SS100304,Maridi,None,None,AMREF Nurse & Midwifery Training,0103000020E6100000010000000A000000F48B12F41772...
3,way/978473724,Comboni Primary School,None,school,None,None,None,None,None,None,...,SS10,Western Equatoria,SS1008,Nzara,SS100802,Nzara Centre,None,None,Comboni Primary School,0103000020E61000000100000005000000753282D77840...
4,node/12158075362,Nanga nursery school,None,school,None,public,None,None,None,None,...,SS10,Western Equatoria,SS1005,Mundri West,SS100503,Kotobi,None,None,Nanga nursery school,0101000020E61000005F419AB168163E407BC6191E569B...



Preview: raw_data.health_facilities
Columns: old_state, state_code, county, county_code, payam, payam_code, site, site_dhis2_name, latitude, longitude
Shape: 5 rows x 10 columns

Preview:


,old_state,state_code,county,county_code,payam,payam_code,site,site_dhis2_name,latitude,longitude
0,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,abiemnom phcc,Abiemnom PHCC,9.398740,28.823400
1,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,awarpiny phcu,Awarpiny PHCU,9.473620,28.888280
2,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,manajoga phcu,Manajoga PHCU,9.392128,28.811239
3,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,panyang phcu,Panyang PHCU,9.409190,28.912590
4,abyei region,SS00,abyei region,SS0001,abyei region,SS000101,abyei civil hospital,Abyei Civil HOSPITAL,9.592052,28.436265



Preview: raw_data.health_facility_type
Columns: State, State_Code, County, County_Code, Payam, Payam_Code , Facility_Name, Type, Facilities_Code, Latitude, Longitude
Shape: 5 rows x 11 columns

Preview:


,State,State_Code,County,County_Code,Payam,Payam_Code,Facility_Name,Type,Facilities_Code,Latitude,Longitude
0,Upper Nile,SS07,Renk,SS0711,Chemmedi,SS071104,Chemmedi PHCC,PHCC,71010101,11.51626,32.97081
1,Upper Nile,SS07,Renk,SS0711,Geger,SS071101,Alaka PHCU,PHCU,71010201,12.16765,32.78522
2,Upper Nile,SS07,Renk,SS0711,Geger,SS071101,Gerger PHCC,PHCC,71010202,11.99025,32.75877
3,Upper Nile,SS07,Renk,SS0711,Geger,SS071101,Jerbana PHCC,PHCC,71010203,12.03827,33.02885
4,Upper Nile,SS07,Renk,SS0711,Geger,SS071101,Romale PHCU,PHCU,71010204,12.07605,32.78196



Preview: raw_data.jiaf_south_sudan_2026
Columns: location, unnamed:_1, unnamed:_2, unnamed:_3, unnamed:_4, unnamed:_5, unnamed:_6, populaton, unnamed:_8, sectoral_pin_(number), unnamed:_10, unnamed:_11, unnamed:_12, unnamed:_13, unnamed:_14, unnamed:_15, unnamed:_16, unnamed:_17, unnamed:_18, unnamed:_19, unnamed:_20
Shape: 5 rows x 21 columns

Preview:


,location,unnamed:_1,unnamed:_2,unnamed:_3,unnamed:_4,unnamed:_5,unnamed:_6,populaton,unnamed:_8,sectoral_pin_(number),...,unnamed:_11,unnamed:_12,unnamed:_13,unnamed:_14,unnamed:_15,unnamed:_16,unnamed:_17,unnamed:_18,unnamed:_19,unnamed:_20
0,Admin 1,Admin 1 P-Code,Admin 2,Admin 2 P-Code,Admin 3,Admin 3 P-Code,Pocket of need,Affected population projection 2026,Population Group,CCCM,...,Nutrition,Food Security,Health,Overarching Protection,Shelter,WASH,Severity,Preliminary PiN,Final PiN,Evidence & Comments
1,Abyei Administrative Area,SS00,Abyei Administrative Area,SS0001,None,None,None,122221.68957757292,IDPs,30000,...,45256.13091560167,75443.06468312298,85561.69897180796,84755,76068.04456226567,47530.65705794502,4,85561.69897180796,85561.69897180796,None
2,Central Equatoria,SS01,Juba,SS0101,None,None,None,185655.53028576757,IDPs,143125,...,55663.994410481486,130641.61559673256,102957.42828930625,125516,156584.91344386805,72574.43456625457,4,156584.91344386805,156584.91344386805,None
3,Central Equatoria,SS01,Kajo-Keji,SS0102,None,None,None,16311.747528959717,IDPs,0,...,5483.031317312983,11295.239262162,8641.764041139364,12061,11514.174726324505,13291.053542115327,4,13291.053542115327,13291.053542115327,None
4,Central Equatoria,SS01,Lainya,SS0103,None,None,None,7434.876377553912,IDPs,0,...,3016.7293663498526,4690.943585002939,3386.0205108896025,5948,4687.204672805727,4405.852668180096,4,5948,5948,None



Preview: raw_data.population_estimates_2024
Columns: admin1, admin1_pcode, admin2, admin2_alternate_name, admin2_pcode, population_-_2025, %_male_children
_under_5, no._of_male
children_under_5, %_female
children_under_5, no._of_female
children_under_5, %_male_children_
aged_5_-_17_years, no._of_male_children_
aged_5_-_17_years, %_female_children_
aged_5_-_17_years, no._of_female_
children_aged_5_-_17_years, %_male_adults_
aged_18_-_60, no._of__male_
adults_aged_18_-_60, %_female_adults_
aged_18_-_60, no._of_female_
adults_aged_18_-_60, %_male_adults_
aged_over_60, no._of_male_adults_
aged_over_60, %_female_adults_
aged_over_60, no._female_adults_
aged_over_60
Shape: 5 rows x 22 columns

Preview:


,admin1,admin1_pcode,admin2,admin2_alternate_name,admin2_pcode,population_-_2025,%_male_children\n_under_5,no._of_male\nchildren_under_5,%_female\nchildren_under_5,no._of_female\nchildren_under_5,...,%_female_children_\naged_5_-_17_years,no._of_female_\nchildren_aged_5_-_17_years,%_male_adults_\naged_18_-_60,no._of__male_\nadults_aged_18_-_60,%_female_adults_\naged_18_-_60,no._of_female_\nadults_aged_18_-_60,%_male_adults_\naged_over_60,no._of_male_adults_\naged_over_60,%_female_adults_\naged_over_60,no._female_adults_\naged_over_60
0,Abyei Administrative Area,SS00,Abyei Administrative Area,Abyei Administrative Area,SS0001,145358.000000,0.0960,13954.368000,0.0890,12936.862000,...,0.1730,25146.934000,0.190,27618.020000,0.202,29362.316000,0.0340,4942.172000,0.0390,5668.962000
1,Central Equatoria,SS01,Juba,None,SS0101,570834.137391,0.0740,42241.726167,0.0974,55599.244982,...,0.1590,90762.627845,0.232,132433.519875,0.241,137571.027111,0.0235,13414.602229,0.0191,10902.932024
2,Central Equatoria,SS01,Kajo-keji,None,SS0102,257989.298823,0.0163,4205.225571,0.0301,7765.477895,...,0.0581,14989.178262,0.328,84620.490014,0.320,82556.575623,0.1270,32764.640951,0.0630,16253.325826
3,Central Equatoria,SS01,Lainya,None,SS0103,119155.101682,0.1050,12511.285677,0.1010,12034.665270,...,0.1040,12392.130575,0.286,34078.359081,0.234,27882.293794,0.0270,3217.187745,0.0330,3932.118356
4,Central Equatoria,SS01,Morobo,None,SS0104,136696.629749,0.0815,11140.775325,0.1050,14353.146124,...,0.1630,22281.550649,0.218,29799.865285,0.203,27749.415839,0.0326,4456.310130,0.0189,2583.566302


In [6]:
# List of tables to preview
tables = [
    'raw_data.education_facilities',
    'raw_data.health_facilities',
    'raw_data.health_facility_type',
    'raw_data.jiaf_south_sudan_2026',
    'raw_data.population_estimates_2024'
]

# Preview each table
for table in tables:
    print(f"\n{'='*60}")
    print(f"Preview: {table}")
    print('='*60)
    
    query = f"SELECT * FROM {table} LIMIT 5;"
    
    with engine.connect() as conn:
        df_preview = pd.read_sql(query, conn)
    
    print(f"Columns: {df_preview.shape[1]}")
    print(f"Rows: {df_preview.shape[0]}")
    print("\nFirst 5 rows:")
    display(df_preview)
    print("\n")


Preview: raw_data.education_facilities
Columns: 22
Rows: 5

First 5 rows:


,id,name,name_en,amenity,building,operator_type,capacity_persons,addr_full,addr_city,source,...,adm1_pcode,adm1_name,adm2_pcode,adm2_name,adm3_pcode,adm3_name,adm4_pcode,adm4_name,name_latin,geometry
0,way/974164633,Africano Mande Academy,None,school,None,None,None,None,Maridi,None,...,SS10,Western Equatoria,SS1003,Maridi,SS100304,Maridi,None,None,Africano Mande Academy,0103000020E61000000100000011000000A62089A8D373...
1,way/974164631,Maridi Teacher Training Institute,None,school,None,None,None,None,Maridi,None,...,SS10,Western Equatoria,SS1003,Maridi,SS100304,Maridi,None,None,Maridi Teacher Training Institute,0103000020E610000001000000070000000A7EC0A84973...
2,way/974164630,AMREF Nurse & Midwifery Training,None,school,None,None,None,None,Maridi,None,...,SS10,Western Equatoria,SS1003,Maridi,SS100304,Maridi,None,None,AMREF Nurse & Midwifery Training,0103000020E6100000010000000A000000F48B12F41772...
3,way/978473724,Comboni Primary School,None,school,None,None,None,None,None,None,...,SS10,Western Equatoria,SS1008,Nzara,SS100802,Nzara Centre,None,None,Comboni Primary School,0103000020E61000000100000005000000753282D77840...
4,node/12158075362,Nanga nursery school,None,school,None,public,None,None,None,None,...,SS10,Western Equatoria,SS1005,Mundri West,SS100503,Kotobi,None,None,Nanga nursery school,0101000020E61000005F419AB168163E407BC6191E569B...





Preview: raw_data.health_facilities
Columns: 10
Rows: 5

First 5 rows:


,old_state,state_code,county,county_code,payam,payam_code,site,site_dhis2_name,latitude,longitude
0,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,abiemnom phcc,Abiemnom PHCC,9.398740,28.823400
1,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,awarpiny phcu,Awarpiny PHCU,9.473620,28.888280
2,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,manajoga phcu,Manajoga PHCU,9.392128,28.811239
3,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,panyang phcu,Panyang PHCU,9.409190,28.912590
4,abyei region,SS00,abyei region,SS0001,abyei region,SS000101,abyei civil hospital,Abyei Civil HOSPITAL,9.592052,28.436265





Preview: raw_data.health_facility_type
Columns: 11
Rows: 5

First 5 rows:


,State,State_Code,County,County_Code,Payam,Payam_Code,Facility_Name,Type,Facilities_Code,Latitude,Longitude
0,Upper Nile,SS07,Renk,SS0711,Chemmedi,SS071104,Chemmedi PHCC,PHCC,71010101,11.51626,32.97081
1,Upper Nile,SS07,Renk,SS0711,Geger,SS071101,Alaka PHCU,PHCU,71010201,12.16765,32.78522
2,Upper Nile,SS07,Renk,SS0711,Geger,SS071101,Gerger PHCC,PHCC,71010202,11.99025,32.75877
3,Upper Nile,SS07,Renk,SS0711,Geger,SS071101,Jerbana PHCC,PHCC,71010203,12.03827,33.02885
4,Upper Nile,SS07,Renk,SS0711,Geger,SS071101,Romale PHCU,PHCU,71010204,12.07605,32.78196





Preview: raw_data.jiaf_south_sudan_2026
Columns: 21
Rows: 5

First 5 rows:


,location,unnamed:_1,unnamed:_2,unnamed:_3,unnamed:_4,unnamed:_5,unnamed:_6,populaton,unnamed:_8,sectoral_pin_(number),...,unnamed:_11,unnamed:_12,unnamed:_13,unnamed:_14,unnamed:_15,unnamed:_16,unnamed:_17,unnamed:_18,unnamed:_19,unnamed:_20
0,Admin 1,Admin 1 P-Code,Admin 2,Admin 2 P-Code,Admin 3,Admin 3 P-Code,Pocket of need,Affected population projection 2026,Population Group,CCCM,...,Nutrition,Food Security,Health,Overarching Protection,Shelter,WASH,Severity,Preliminary PiN,Final PiN,Evidence & Comments
1,Abyei Administrative Area,SS00,Abyei Administrative Area,SS0001,None,None,None,122221.68957757292,IDPs,30000,...,45256.13091560167,75443.06468312298,85561.69897180796,84755,76068.04456226567,47530.65705794502,4,85561.69897180796,85561.69897180796,None
2,Central Equatoria,SS01,Juba,SS0101,None,None,None,185655.53028576757,IDPs,143125,...,55663.994410481486,130641.61559673256,102957.42828930625,125516,156584.91344386805,72574.43456625457,4,156584.91344386805,156584.91344386805,None
3,Central Equatoria,SS01,Kajo-Keji,SS0102,None,None,None,16311.747528959717,IDPs,0,...,5483.031317312983,11295.239262162,8641.764041139364,12061,11514.174726324505,13291.053542115327,4,13291.053542115327,13291.053542115327,None
4,Central Equatoria,SS01,Lainya,SS0103,None,None,None,7434.876377553912,IDPs,0,...,3016.7293663498526,4690.943585002939,3386.0205108896025,5948,4687.204672805727,4405.852668180096,4,5948,5948,None





Preview: raw_data.population_estimates_2024
Columns: 22
Rows: 5

First 5 rows:


,admin1,admin1_pcode,admin2,admin2_alternate_name,admin2_pcode,population_-_2025,%_male_children\n_under_5,no._of_male\nchildren_under_5,%_female\nchildren_under_5,no._of_female\nchildren_under_5,...,%_female_children_\naged_5_-_17_years,no._of_female_\nchildren_aged_5_-_17_years,%_male_adults_\naged_18_-_60,no._of__male_\nadults_aged_18_-_60,%_female_adults_\naged_18_-_60,no._of_female_\nadults_aged_18_-_60,%_male_adults_\naged_over_60,no._of_male_adults_\naged_over_60,%_female_adults_\naged_over_60,no._female_adults_\naged_over_60
0,Abyei Administrative Area,SS00,Abyei Administrative Area,Abyei Administrative Area,SS0001,145358.000000,0.0960,13954.368000,0.0890,12936.862000,...,0.1730,25146.934000,0.190,27618.020000,0.202,29362.316000,0.0340,4942.172000,0.0390,5668.962000
1,Central Equatoria,SS01,Juba,None,SS0101,570834.137391,0.0740,42241.726167,0.0974,55599.244982,...,0.1590,90762.627845,0.232,132433.519875,0.241,137571.027111,0.0235,13414.602229,0.0191,10902.932024
2,Central Equatoria,SS01,Kajo-keji,None,SS0102,257989.298823,0.0163,4205.225571,0.0301,7765.477895,...,0.0581,14989.178262,0.328,84620.490014,0.320,82556.575623,0.1270,32764.640951,0.0630,16253.325826
3,Central Equatoria,SS01,Lainya,None,SS0103,119155.101682,0.1050,12511.285677,0.1010,12034.665270,...,0.1040,12392.130575,0.286,34078.359081,0.234,27882.293794,0.0270,3217.187745,0.0330,3932.118356
4,Central Equatoria,SS01,Morobo,None,SS0104,136696.629749,0.0815,11140.775325,0.1050,14353.146124,...,0.1630,22281.550649,0.218,29799.865285,0.203,27749.415839,0.0326,4456.310130,0.0189,2583.566302
